# Introspection in a small transformer — a tutorial reproductionA hands-on walkthrough of the experiment in this repository. Every cell calls afunction from `src/`; nothing is reimplemented here. If a cell surprises you,open the module it calls and read it.**Runtime:** roughly 10–30 minutes on an Apple-Silicon MacBook, or a couple ofminutes with `MODEL = "random-tiny"` (which downloads nothing and producesmeaningless numbers — a plumbing check only).

## 1. IntroductionLindsey (2025), *Emergent Introspective Awareness in Language Models*([transformer-circuits.pub/2025/introspection](https://transformer-circuits.pub/2025/introspection/index.html)),asks whether a language model has any access to its own internal state.The method is **concept injection**. A direction in the residual stream thatrepresents some concept is extracted, then added back into the residual streamduring an unrelated conversation. The model is asked whether it notices anythingunusual — *before* it has said anything about the concept. In Claude Opus 4.1 themodel sometimes says yes and correctly names the concept, roughly 20% of the timeat the best layer.

## 2. What we are testing**H0.** With the injection layer, strength, and prompt fixed in advance, injectinga concept direction changes the model's Yes/No detection response no more than anorm-matched random vector does, and the model identifies the injected concept atchance.**H1.** Injecting a concept direction raises the detection signal above thenorm-matched random control *and* the model identifies the concept above chance,while an unrelated no-answer question is unaffected.The random control is the load-bearing one. *Any* perturbation of the residualstream changes the output. Only a **content-specific** change is evidence aboutintrospection.

## 3. Original experiment vs this reproduction| | Lindsey (2025) | This repo ||---|---|---|| model | Claude Opus 4.1 (frontier scale) | Qwen2.5-0.5B-Instruct || grading | LLM judge on free text | restricted logit readout + forced choice || concepts | large, varied | 30, balanced across categories || strength | raw multiples of the vector | multiples of the layer's residual norm || layers | evenly spaced | every layer |This is a **conceptual replication**, not a replication. It shares the causallogic and the controls; it does not share the scale, the model, or the gradingprocedure, and it cannot confirm or refute the original result.

## 4. Setup

In [ ]:
import sys, pathlibsys.path.insert(0, str(pathlib.Path.cwd().parent))import pandas as pdimport torchfrom src.runner import load_config, build_contextfrom src.device import set_seedconfig = load_config("../config.yaml")# Switch to "random-tiny" for a no-download plumbing check.# config["model"]["name"] = "random-tiny"# Keep the notebook fast; run_all.py uses the full design.config["design"]["n_concepts"] = 12config["intervention"]["n_concepts"] = 4config["generation"]["n_examples"] = 3set_seed(config["seed"])config["model"]["name"], config["seed"]

## 5. Load the model`build_context` loads the model, extracts a concept vector for every concept atevery layer, measures the residual-stream norm that sets the injection scale, andresolves the "Yes"/"No" token ids. The concept-vector extraction is the slow part(one forward pass per word per layer).

In [ ]:
ctx, trials = build_context(config)print(f"layers            {ctx.n_layers}")print(f"confirmatory layer {ctx.primary_layer}")print(f"concepts          {len(ctx.concept_vectors)}")print(f"trials            {len(trials)}")ctx.summary

## 6. Transformer forward-pass walkthroughBefore claiming anything about internal states, look at them. This prints everytensor in one forward pass, verifies the residual-stream identity numerically(`resid_pre + attn_out == resid_mid`, `resid_mid + mlp_out == resid_post`), andshows how the residual norm grows with depth — which is why injection strength inthis repo is measured in residual norms rather than raw multiples.

In [ ]:
from src.transformer_walkthrough import walkthroughout = walkthrough(ctx.model, layer=ctx.primary_layer)None

## 7. Generate experimental promptsThe trial set is crossed: every concept appears in every condition, so the armsare matched item-for-item. Note that the concept word appears **nowhere** in thedetection prompt.

In [ ]:
from src.prompts import detection_messages, identification_messages, CONCEPT_WORDSfor msg in detection_messages():    print(f"[{msg['role']:>9}] {msg['content']}\n")print("conditions:", sorted({t.condition for t in trials}))print("question kinds:", sorted({t.question_kind for t in trials}))pd.DataFrame([t.__dict__ for t in trials]).head(6)

## 8. Behavioural baselineStrength 0 — the untouched model. Two things to check before going further:* what the model's **default** answer is (if it already says "Yes", every later  comparison is relative to that, not to zero);* whether forced-choice identification is **at chance** with nothing injected.  If it is not, the prompt is leaking the answer and the experiment is invalid.

In [ ]:
from src.introspection import run_trialsbaseline = pd.DataFrame(run_trials(ctx, trials, layer=ctx.primary_layer, strength=0.0))det = baseline[baseline.question_kind == "detection"]ident = baseline[baseline.question_kind == "identification"]print(f"default yes_minus_no      {det.yes_minus_no.mean():+.3f}")print(f"fraction answering 'Yes'  {(det.yes_minus_no > 0).mean():.2f}")print(f"identification accuracy   {ident.id_correct.mean():.3f}  (chance {ident.chance.iloc[0]:.3f})")

## 9. Cache activations and build a concept vector`v_word = act("Tell me about {word}") - mean(act over 50 filler words)`.Subtracting the filler baseline is what turns "the activation while describingsomething" into "the activation specific to *this* concept". Without it thevector mostly encodes the shared prompt template.

In [ ]:
from src import hooks as hkconcept = list(ctx.concept_vectors)[0]layer = ctx.primary_layerv = ctx.concept_vectors[concept][layer]coeff = hk.injection_coefficient(v, config["injection"]["primary_strength"], ctx.norm_units[layer])print(f"concept        {concept}")print(f"vector shape   {tuple(v.shape)}")print(f"||v||          {v.norm():.3f}")print(f"norm unit      {ctx.norm_units[layer]:.3f}")print(f"coefficient    {coeff:.4f}  ->  we add {coeff * v.norm():.3f} = "      f"{config['injection']['primary_strength']} residual norms")

## 10. Inspect internal stateWhere does the injection land, and how far does it push the forward pass? Ahealthy intervention drops the cosine similarity sharply at the injection siteand then partially recovers. A curve stuck near zero means the model was broken,not nudged, and nothing measured under those conditions is interpretable.

In [ ]:
from src.introspection import residual_similaritysim = pd.DataFrame(residual_similarity(    ctx, concept, inject_layer=layer, strength=config["injection"]["primary_strength"]))sim[["read_layer", "cosine_similarity", "clean_norm", "injected_norm"]]

## 11. Introspection testThe confirmatory measurement: injected vs norm-matched random control at thepre-registered layer and strength.

In [ ]:
strength = config["injection"]["primary_strength"]primary = pd.DataFrame(run_trials(ctx, trials, layer=layer, strength=strength))primary.groupby(["condition", "question_kind"])[["yes_minus_no"]].mean()

## 12. ControlsThree of them, each aimed at a specific failure mode:| control | failure mode it detects ||---|---|| norm-matched random vector | the model reacts to *being perturbed*, not to content || no injection | the model says "Yes" by default; a floor/ceiling problem || yes-bias question | the injection makes the model generally more agreeable |The yes-bias control is compared **within question type** — the same unrelatedquestion with and without injection — because comparing it against the detectionquestion would confound "which question" with "was anything injected".

In [ ]:
from src.metrics import detection_table, identification_table, yesbias_tabledet_table = detection_table(primary, reference="random_control",                            n_boot=2000, n_perm=5000, seed=config["seed"])ident_table = identification_table(primary, seed=config["seed"], n_boot=2000)yb_table = yesbias_table(primary, baseline, n_perm=5000, n_boot=2000, seed=config["seed"])display(det_table)display(ident_table)display(yb_table)

## 13. InterventionThe injection already shows *that* the residual stream matters. The ablationsweep asks *how* the signal travels: does any downstream component read it, ordoes the vector ride the residual skip connection straight into the logits?The reported quantity is a double subtraction, because ablating a componentdamages the model whether or not anything was injected.

In [ ]:
from src.interventions import ablation_sweepablation = pd.DataFrame(ablation_sweep(    ctx, trials, layer=layer, strength=strength,    n_concepts=config["intervention"]["n_concepts"],))(ablation.groupby(["component", "ablate_layer"])["interaction"]         .mean().sort_values().head(10).to_frame("mean_interaction"))

## 14. Quantitative resultsLayer sweep — **exploratory**, Holm-corrected across layers. The confirmatoryclaim rests on the pre-registered layer alone; reading the best layer off thistable post hoc is how a null becomes a false positive.

In [ ]:
from src.introspection import layer_sweep, logit_lens_by_layerfrom src.metrics import layer_sweep_tablesweep = pd.DataFrame(layer_sweep(ctx, trials, strength=strength))ls_table = layer_sweep_table(sweep, n_perm=2000, seed=config["seed"])ls_table

## 15. VisualizationFigures land in `results/figures/`, with an interpretation for each in`results/figures/CAPTIONS.md`.

In [ ]:
from src.visualization import make_all_figuresfrom IPython.display import Image, display as showdistractors = {t.concept: t.distractors for t in trials               if t.question_kind == "identification" and t.condition == "injected"}lens_rows = []for c in list(ctx.concept_vectors)[: config["intervention"]["n_concepts"]]:    lens_rows += logit_lens_by_layer(ctx, c, inject_layer=layer, strength=strength,                                     distractors=distractors.get(c, []))lens = pd.DataFrame(lens_rows)paths = make_all_figures(    {"primary": primary, "layer_sweep": sweep, "logit_lens": lens,     "residual_similarity": sim, "ablation": ablation},    "../results/figures", primary_layer=layer, seed=config["seed"],)for p in paths:    show(Image(filename=str(p)))

## 16. InterpretationWork through these in order. Stop at the first one that fails.1. **Is the measure usable?** Baseline identification at chance, and the model   not already saying "Yes" to everything.2. **Is the intervention sane?** Cosine similarity recovers after the injection   site rather than collapsing.3. **Is the effect content-specific?** Injected > norm-matched random, paired,   at the pre-registered layer.4. **Does the effect survive the yes-bias control?** If the same injection moves   an unrelated no-answer question by a comparable amount, it is a general   agreeableness shift, not detection.5. **Can the model name what was injected?** Forced choice above chance.6. **Is the mechanism plausible?** The concept becomes readable in the logit lens   *before* the Yes/No decision forms, and some downstream component's ablation   removes the effect.Only 1–6 together support anything like introspection. 3 alone does not.

## 17. Limitations* **Scale.** ~0.5B parameters against frontier models. The original reports the  effect at ~20% even in Opus 4.1; a null here is uninformative about that claim.* **Not the same model.** Qwen2.5 has a different architecture, tokenizer, and  post-training pipeline. Introspective self-report is plausibly a  post-training-dependent capability.* **Restricted readout.** Scoring Yes/No logits and forced choice is  deterministic and cheap, but it is not what the paper graded. A model could  fail this readout and still produce a creditable free-text report, or pass it  without producing one.* **Injection over the question tokens.** The default window covers the trial  question. Re-run with `config["injection"]["window"] = "answer"` to check the  model is not simply reacting to a corrupted reading of the question.* **Multiple comparisons.** The layer sweep is exploratory and corrected; the  strength sweep is not corrected at all and should be read as descriptive.

## 18. Further experiments1. Repeat at Qwen2.5-1.5B-Instruct and 3B, and plot the effect against parameter   count. Scale is the most likely explanation for a null.2. Swap the readout for an LLM judge on free text, matching the original grading   procedure, and see whether the conclusion changes.3. Extract concept vectors from contrastive *pairs* rather than a filler-word   mean, as the original also does, and compare vector quality.4. Reproduce the paper's second experiment: inject while the model transcribes   unrelated text, and test whether it keeps "thought" and "text" apart.5. Reproduce the prefill experiment: force an out-of-character output, then   retroactively inject the matching concept beforehand, and test whether the   model stops disavowing it.